# Notebook 04: OD Matrix Construction

Builds the origin-destination matrix from XDR cell-to-cell transitions and decomposes intra-site vs inter-site mobility.

## Setup

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import h3
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

DATA    = Path("../data")
FIGURES = Path("../figures")
FIGURES.mkdir(parents=True, exist_ok=True)
XDR_DIR = DATA / "xdr_exports"
CATALOG = DATA / "tower_catalog.csv"
CENSUS  = DATA / "Cartografia_censo2024_R13/Cartografia_censo2024_R13.gpkg"

CRS_GEO = "EPSG:4674"
CRS_UTM = "EPSG:32719"

## Load XDR transition data

In [ ]:
td_cell_od = pd.read_parquet(XDR_DIR / "td_cell_od")
td_cell_od["is_intrasite"] = td_cell_od["is_intrasite"].fillna(False).astype(bool)

print(f"td_cell_od: {len(td_cell_od):,} rows")
print(f"  Total trips: {td_cell_od['n_trips'].sum():,}")

td_bts_od = pd.read_parquet(XDR_DIR / "td_bts_od")
print(f"td_bts_od: {len(td_bts_od):,} rows")

## Join with tower catalog

In [ ]:
cell_lookup = pd.read_parquet(DATA / "cell_lookup.parquet")
bts = pd.read_parquet(DATA / "bts_sites.parquet")

# Load catalog for azimuths
cat_raw = pd.read_csv(CATALOG)
cat_r13 = cat_raw[cat_raw["admin_code"].astype(str).str.startswith("13")].copy()

cell_az = cat_r13[["cell_id", "azimuth"]].dropna(subset=["azimuth"]).copy()
cell_az["cell_id"] = cell_az["cell_id"].astype(int).astype(str)

cl = cell_lookup.copy()
cl["cell_id"] = cl["cell_id"].astype(int).astype(str)
cell_info = cl.merge(cell_az, on="cell_id", how="left")

cell_info = cell_info.merge(
    bts[["bts_id", "radius", "n_sectors", "lat", "lon"]].rename(
        columns={"bts_id": "site_id"}
    ),
    on="site_id", how="left",
)

cell_info = cell_info.drop_duplicates(subset=["cell_id"], keep="first")
print(f"Cell info table: {len(cell_info):,} rows")

## Join cell info to OD transitions

In [ ]:
td_cell_od["cell_from"] = td_cell_od["cell_from"].astype(str)
td_cell_od["cell_to"]   = td_cell_od["cell_to"].astype(str)

join_cols = ["cell_id", "site_id", "azimuth", "radius", "n_sectors"]

od = td_cell_od.merge(
    cell_info[join_cols].rename(columns={c: c + "_from" for c in join_cols if c != "cell_id"}),
    left_on="cell_from", right_on="cell_id", how="inner",
).drop(columns=["cell_id"])

od = od.merge(
    cell_info[join_cols].rename(columns={c: c + "_to" for c in join_cols if c != "cell_id"}),
    left_on="cell_to", right_on="cell_id", how="inner",
).drop(columns=["cell_id"])

match_rate = len(od) / len(td_cell_od)
print(f"Matched {len(od):,} / {len(td_cell_od):,} transitions ({match_rate:.1%})")
print(f"  Trips matched: {od['n_trips'].sum():,} / {td_cell_od['n_trips'].sum():,}")

## Intra-site trip classification

In [ ]:
od["is_intrasite_site"] = (od["site_id_from"] == od["site_id_to"])

intra_trips = od.loc[od["is_intrasite_site"], "n_trips"].sum()
inter_trips = od.loc[~od["is_intrasite_site"], "n_trips"].sum()
total_trips = od["n_trips"].sum()

print(f"=== Trip classification (by merged site) ===")
print(f"Intra-site: {intra_trips:,} ({intra_trips/total_trips:.1%})")
print(f"Inter-site: {inter_trips:,} ({inter_trips/total_trips:.1%})")
print(f"Total:      {total_trips:,}")

## Displacement estimation

In [ ]:
def haversine_m(lat1, lon1, lat2, lon2):
    R = 6_371_000
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# Inter-site: tower-to-tower distance
od["displacement_m"] = haversine_m(
    od["lat_from"], od["lon_from"], od["lat_to"], od["lon_to"],
)

# Intra-site: sector-crossing vs technology handoff
mask_intra = od["is_intrasite_site"]
az_diff = np.abs(od.loc[mask_intra, "azimuth_from"] - od.loc[mask_intra, "azimuth_to"])
az_diff = np.minimum(az_diff, 360 - az_diff)

AZIMUTH_TOL = 5
od["is_tech_handoff"] = False
od.loc[mask_intra, "is_tech_handoff"] = az_diff < AZIMUTH_TOL

n_handoff = od.loc[od["is_tech_handoff"], "n_trips"].sum()
n_sector_cross = od.loc[mask_intra & ~od["is_tech_handoff"], "n_trips"].sum()
print(f"Intra-site breakdown:")
print(f"  Sector-crossing:    {n_sector_cross:,} trips")
print(f"  Technology handoff: {n_handoff:,} trips (excluded)")

# Estimate displacement for sector crossings
mask_cross = mask_intra & ~od["is_tech_handoff"]
az_diff_cross = np.abs(od.loc[mask_cross, "azimuth_from"] - od.loc[mask_cross, "azimuth_to"])
az_diff_cross = np.minimum(az_diff_cross, 360 - az_diff_cross)
az_diff_rad = np.deg2rad(az_diff_cross)

r_from = od.loc[mask_cross, "radius_from"]
od.loc[mask_cross, "displacement_m"] = (2 * r_from / 3) * 2 * np.sin(az_diff_rad / 2)
od.loc[od["is_tech_handoff"], "displacement_m"] = np.nan

valid = od[~od["is_tech_handoff"]]
print(f"\nMedian displacement (m):")
print(f"  Inter-site: {valid.loc[~valid['is_intrasite_site'], 'displacement_m'].median():,.0f}")
print(f"  Intra-site: {valid.loc[valid['is_intrasite_site'], 'displacement_m'].median():,.0f}")

## Figure: Displacement distribution (Fig. 5 in paper)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
bins = np.logspace(1, 5, 60)

inter = od[~od["is_intrasite_site"]]
all_valid = od[~od["is_tech_handoff"]].dropna(subset=["displacement_m"])

ax.hist(inter["displacement_m"].clip(10),
        bins=bins, weights=inter["n_trips"],
        alpha=0.5, label="Inter-site only", color="#e41a1c")
ax.hist(all_valid["displacement_m"].clip(10),
        bins=bins, weights=all_valid["n_trips"],
        alpha=0.5, label="All trips (incl. sector-crossing)", color="#377eb8")
ax.set_xscale("log")
ax.set_xlabel("Displacement (m)")
ax.set_ylabel("Number of trips")
ax.set_title("Displacement distribution: effect of including intra-site sector crossings")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "fig_displacement_distribution.pdf", bbox_inches="tight")
plt.show()

## Aggregate OD to H3 and comuna level

In [ ]:
H3_RES = 8

od["h3_from"] = [
    h3.latlng_to_cell(lat, lon, H3_RES)
    for lat, lon in zip(od["lat_from"], od["lon_from"])
]
od["h3_to"] = [
    h3.latlng_to_cell(lat, lon, H3_RES)
    for lat, lon in zip(od["lat_to"], od["lon_to"])
]

od_h3 = (
    od.groupby(["h3_from", "h3_to"])
    .agg(n_trips=("n_trips", "sum"), n_users=("n_users", "sum"))
    .reset_index()
)
print(f"H3 OD matrix: {len(od_h3):,} pairs")

od["codcom_from"] = od["codcom_from"].astype(int)
od["codcom_to"]   = od["codcom_to"].astype(int)

def weighted_mean_displacement(group):
    d = group["displacement_m"]
    w = group["n_trips"]
    mask = d.notna()
    if mask.sum() == 0:
        return np.nan
    return np.average(d[mask], weights=w[mask])

od_comuna = (
    od.groupby(["codcom_from", "codcom_to"])
    .apply(lambda g: pd.Series({
        "n_trips": g["n_trips"].sum(),
        "n_users": g["n_users"].sum(),
        "mean_displacement": weighted_mean_displacement(g),
        "intra_site_trips": g.loc[g["is_intrasite_site"], "n_trips"].sum(),
        "tech_handoff_trips": g.loc[g["is_tech_handoff"], "n_trips"].sum(),
    }))
    .reset_index()
)
od_comuna["n_trips"] = od_comuna["n_trips"].astype(int)
od_comuna["n_users"] = od_comuna["n_users"].astype(int)

print(f"Comuna OD matrix: {len(od_comuna):,} pairs")
print(f"  Total trips: {od_comuna['n_trips'].sum():,}")

## Save outputs

In [ ]:
od_h3.to_parquet(DATA / "od_h3.parquet", index=False)
print(f"Saved: od_h3.parquet ({len(od_h3):,} rows)")

od_comuna.to_parquet(DATA / "od_comuna.parquet", index=False)
print(f"Saved: od_comuna.parquet ({len(od_comuna):,} rows)")

od_for_ipw = od[~od["is_tech_handoff"]].dropna(subset=["displacement_m"])
od_for_ipw = od_for_ipw[["cell_from", "cell_to", "site_id_from", "site_id_to",
                          "codcom_from", "codcom_to", "h3_from", "h3_to",
                          "is_intrasite_site", "n_trips", "n_users",
                          "displacement_m", "radius_from", "n_sectors_from"]].copy()
od_for_ipw.to_parquet(DATA / "od_for_ipw.parquet", index=False)
print(f"Saved: od_for_ipw.parquet ({len(od_for_ipw):,} rows)")